In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler,normalize
import pyarrow.parquet as pq

import math
import gc

In [ ]:
cutoff_date = pd.Timestamp('2021-10-19', tz= 'UTC')#'2022-1-2'

In [25]:
chunks = pd.read_csv(
    "bitcoin_tweets_clean_bert.csv",
    chunksize=100_000,
    engine="python",          # 🔴 extrem wichtig
)

In [26]:
df = pd.concat(chunks)

In [27]:
len(df['username'].unique())

2055355

In [28]:
df.columns

Index(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'datetime', 'username',
       'text'],
      dtype='object')

In [29]:
df['datetime'] = pd.to_datetime(df['datetime'])

In [30]:
df= df[df['datetime'] <= cutoff_date]

In [31]:
df_names = df[['username']]

In [32]:
len(df_names['username'].unique())

1737004

In [33]:
del df
gc.collect()

112

In [3]:
chunk_files = list(Path("Embedings/").glob("embeddings_chunk_*.parquet"))

In [4]:
user_means = {}
user_counts = {}

for f in chunk_files:
    print(f"Lade {f.name}")
    df_chunk = pd.read_parquet(f)

    if len(df_chunk.columns) == 1026:
        df_chunk['datetime'] = pd.to_datetime(df_chunk['datetime'])
        mask = df_chunk['datetime'] <= cutoff_date
        df_chunk = df_chunk.loc[mask]
        df_chunk = df_chunk.drop(columns=['datetime'])
        df_chunk = df_chunk.rename(columns={'username': 'name'})
    else:
        print("incorrect col length")
        continue

    # Nur Embedding-Spalten einmal bestimmen
    emb_cols = [c for c in df_chunk.columns if c != "name"]

    for name, group in df_chunk.groupby("name", sort=False):

        # Direkt float32, KEIN stack
        X = group[emb_cols].to_numpy(dtype=np.float32, copy=False)

        # Mittelwert dieses Chunks für den User
        chunk_mean = X.mean(axis=0)        # float32
        chunk_count = len(X)

        if name in user_means:
            prev_mean = user_means[name]
            prev_count = user_counts[name]

            total_count = prev_count + chunk_count

            # Numerisch stabiler inkrementeller Mittelwert
            user_means[name] = (
                prev_mean
                + (chunk_mean - prev_mean) * (chunk_count / total_count)
            )

            user_counts[name] = total_count
        else:
            user_means[name] = chunk_mean
            user_counts[name] = chunk_count

        # Wichtig: Referenz freigeben
        del X, chunk_mean

    del df_chunk

Lade embeddings_chunk_0.parquet
Lade embeddings_chunk_1.parquet
Lade embeddings_chunk_10.parquet
Lade embeddings_chunk_100.parquet
Lade embeddings_chunk_101.parquet
Lade embeddings_chunk_102.parquet
Lade embeddings_chunk_103.parquet
Lade embeddings_chunk_104.parquet
Lade embeddings_chunk_105.parquet
Lade embeddings_chunk_106.parquet
Lade embeddings_chunk_107.parquet
Lade embeddings_chunk_108.parquet
Lade embeddings_chunk_109.parquet
Lade embeddings_chunk_11.parquet
Lade embeddings_chunk_110.parquet
Lade embeddings_chunk_111.parquet
Lade embeddings_chunk_112.parquet
Lade embeddings_chunk_113.parquet
Lade embeddings_chunk_114.parquet
Lade embeddings_chunk_115.parquet
Lade embeddings_chunk_116.parquet
Lade embeddings_chunk_117.parquet
Lade embeddings_chunk_118.parquet
Lade embeddings_chunk_119.parquet
Lade embeddings_chunk_12.parquet
Lade embeddings_chunk_120.parquet
Lade embeddings_chunk_121.parquet
Lade embeddings_chunk_122.parquet
Lade embeddings_chunk_123.parquet
Lade embeddings_chunk

In [5]:
problematic = {}

for k, arr in user_means.items():
    # np.ravel flacht das Array ab, falls es multidimensional ist
    bad_elements = [x for x in np.ravel(arr) if not np.issubdtype(type(x), np.number)]
    if bad_elements:  # nur speichern, wenn etwas gefunden wurde
        problematic[k] = bad_elements

In [6]:
problematic

{}

In [7]:
del problematic
del chunk_files
del emb_cols
del prev_mean

gc.collect()

48

In [8]:
dict(list(user_counts.items())[:100])

{'BitcoinBellyB': 6539,
 'BTCMEXGlobal': 6,
 'CryptoNetDaily': 3991,
 'Bitstamp': 133,
 'btcusdalerts': 418,
 'BitcoinAverage': 2045,
 'K_9Girl': 10,
 'BTCtoCAD': 9662,
 'WSJ': 231,
 'coindata_uk': 4259,
 'fixtheblock': 12540,
 'awesomepersona2': 149,
 'JeremyKissel': 314,
 'cryptowat_ch': 90,
 'vpaul8483': 1,
 'littleprinceJD': 5175,
 'abubakarhayus': 1,
 'RealTradeAlts': 18,
 'zloadr': 4015,
 'amneesiack': 6,
 'crypto_buy_tips': 5961,
 'StephenPKendal': 379,
 'VolumeMonitor': 6768,
 'XCryptoRTaeP': 8,
 'bmurphypointman': 477433,
 'Jimmmyz169': 88,
 'nic__carter': 1059,
 'heatrockz': 73,
 'JacobCanfield': 472,
 'WallStSenpai': 225,
 'ledgerstatus': 157,
 'yackzilon': 19,
 'bot_blockchain': 363,
 'ccinfonet': 1068,
 'BtcPulse': 8526,
 'HappyTours5': 530,
 'therealpharmG': 2,
 'day_bitcoin': 2457,
 'Atilaeus': 1,
 'CryptoEliteNews': 5229,
 'RookieXBT': 266,
 'jccrypto2': 55,
 'Crypto_Salmon': 129,
 'Bulldozer0': 421,
 'swaythebaba': 5,
 'BrianVarnerVA': 48,
 'milmaspop': 512,
 'Christea

In [9]:
len(user_counts)

2055355

In [10]:
len(user_means)

2055355

In [11]:
dict(list(user_means.items())[:1])

{'BitcoinBellyB': array([-0.00390503,  0.00490928, -0.01953064, ...,  0.00638022,
        -0.00328811,  0.00266207], dtype=float32)}

In [12]:
for k in user_means:
    user_means[k] = user_means[k].astype(np.float16, copy=False)

In [ ]:
np.savez_compressed(
    "user_means_and_counts.npz",
    user_means=user_means,
    user_counts=user_counts
)

NameError: name 'user_means' is not defined

In [ ]:
data = np.load("user_means_and_counts_total.npz", allow_pickle=True)

user_means = data["user_means"].item()
user_counts = data["user_counts"].item()

In [5]:
def normalize_embeddings_inplace(user_means, batch_size=10000):
    keys = list(user_means.keys())
    n = len(keys)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_keys = keys[start:end]

        X = np.array(
            [user_means[k] for k in batch_keys],
            dtype=np.float32
        )

        X_norm = normalize(X, norm="l2", axis=1, copy= False)

        for i, k in enumerate(batch_keys):
            user_means[k] = X_norm[i].copy()

        # Wichtig: Referenzen freigeben
        del X, X_norm, batch_keys, end

In [6]:
normalize_embeddings_inplace(user_means, batch_size=2500)

In [ ]:
np.savez_compressed(
    "user_means_norm_and_counts.npz",
    user_means=user_means,
    user_counts=user_counts
)